# Lección 8.5 — Visualización: t-SNE y UMAP, el plano de mesas de una boda

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 8.4 (PCA), Lección 8.2 (K-means)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué conserva t-SNE (los **vecinos**) y qué **no** conserva (tamaños y distancias entre grupos).
- Comparar t-SNE con PCA y saber cuándo usar cada uno.
- Predecir el efecto de `perplexity` (t-SNE) y de `n_neighbors` / `min_dist` (UMAP).
- Leer un gráfico de t-SNE **sin caer en sus trampas**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.
>
> ⏳ t-SNE es más lento que PCA: algunas celdas tardan entre 10 y 40 segundos. Es normal.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_blobs
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

plt.rcParams["figure.figsize"] = (8, 4.5)

digitos = load_digits()
X_d, y_d = digitos.data, digitos.target

def mapa(ax, X2, colores, titulo, cmap="tab10"):
    ax.scatter(*X2.T, c=colores, cmap=cmap, s=6)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

print("✅ Todo listo")

---
## 1. 📖 La intuición: el plano de mesas de una boda 💒

Organizas una boda con 200 invitados y 20 mesas. Tu objetivo: que **cada invitado quede junto a la gente que conoce**. Los primos con los primos, los compañeros de trabajo con los compañeros de trabajo.

Fíjate en lo que **sí** te importa y lo que **no**:
- ✅ **Sí:** que los que se conocen queden **en la misma mesa** (los vecinos se conservan).
- ❌ **No:** que la mesa de los primos esté a 3 metros o a 10 de la mesa del trabajo. La sala tiene que acomodarse como se pueda.
- ❌ **No:** que una mesa sea más grande que otra porque "ese grupo es más diverso". Todas las mesas son del mismo tamaño.

**t-SNE** (*t-distributed Stochastic Neighbor Embedding*) hace ese plano de mesas con datos: coloca cada punto en un mapa 2D de forma que **sus vecinos de verdad queden cerca**. Lo demás (distancias entre grupos, tamaño de los grupos) lo sacrifica.

### ✋ Ejercicio 1 — Calentamiento
Ves un plano de mesas de la boda: la mesa A está pegada a la mesa B, y la mesa C está en la otra punta del salón. ¿Puedes concluir que los invitados de A se parecen más a los de B que a los de C?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**No con seguridad.** El organizador solo se preocupó de que **dentro** de cada mesa estuvieran juntos los conocidos. Dónde quedó cada mesa en el salón depende de cómo cupieron. Es exactamente la trampa más común al leer un gráfico de t-SNE: **las distancias entre grupos no son confiables**.
</details>

---
## 2. 📖 El concepto: PCA vs. t-SNE

En la Lección 8.4, PCA dibujó los dígitos en 2D y muchos se mezclaban en el centro. PCA es una "foto" **lineal**: solo puede girar y aplastar los datos. t-SNE, en cambio, puede **doblar y estirar** el espacio para que los vecinos queden juntos.

| | PCA | t-SNE |
|---|---|---|
| Qué conserva | Las direcciones de **mayor varianza** (la forma global) | Los **vecinos** de cada punto (la estructura local) |
| Tipo | Lineal (girar y proyectar) | No lineal (doblar y estirar) |
| Determinista | Sí | No: cambia con la semilla |
| Datos nuevos | `transform` ✅ | No tiene `transform` ❌ |
| Velocidad | Instantáneo | Lento con muchos datos |
| Para qué | Comprimir, preprocesar, primera mirada | **Solo para visualizar** |

### 🔮 Ejercicio 2
Vamos a dibujar los 1,797 dígitos con PCA y con t-SNE, lado a lado. ¿Cuántos grupos esperas distinguir en cada gráfico?

✍️ *Mi apuesta:*

In [ ]:
X_pca = PCA(n_components=2).fit_transform(X_d)
X_tsne = TSNE(n_components=2, perplexity=30, random_state=0).fit_transform(X_d)    # ⏳ ~15 s

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
mapa(axes[0], X_pca, y_d, "PCA")
mapa(axes[1], X_tsne, y_d, "t-SNE (perplexity=30)")
for d in range(10):                                   # escribimos cada dígito en el centro de su nube
    axes[1].text(*np.median(X_tsne[y_d == d], axis=0), str(d), fontsize=16, weight="bold")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **PCA:** se distinguen 3–4 zonas; la mayoría de los dígitos se amontonan.
- **t-SNE:** aparece una isla bien separada por cada dígito, ¡y t-SNE **nunca vio** las etiquetas! Hay detalles interesantes: el **1** queda partido en dos o tres islitas (hay varias formas de escribir un 1, con y sin "gorrito" o base) y unos pocos puntos caen en islas ajenas (dígitos mal escritos).

Esto es lo que hace famoso a t-SNE: revela estructura que PCA no puede mostrar en 2D.
</details>

---
## 3. 📖 Cómo funciona por dentro (sin fórmulas)

1. **En el espacio original** (64 dimensiones), cada punto reparte "puntos de amistad" entre los demás: mucho a los muy cercanos, casi nada a los lejanos. La **perplejidad** (`perplexity`) dice aproximadamente **con cuántos vecinos** se preocupa cada punto.
2. **En el mapa 2D**, coloca todos los puntos (al principio, a partir de PCA o al azar).
3. **Mueve los puntos poco a poco** (como el descenso de gradiente, Lección 3.3): los amigos se **atraen** como resortes y los desconocidos se **repelen**, hasta que las amistades del mapa se parecen a las del espacio original.

La "t" del nombre es un truco: en 2D usa una curva con **colas largas** que deja que los grupos se separen con generosidad. Por eso las islas de t-SNE se ven tan limpias… y por eso los espacios entre islas **no significan nada preciso**.

### ✋ Ejercicio 3
Según el paso 3, t-SNE "mueve los puntos poco a poco" desde una posición inicial. ¿Qué crees que pasará si lo ejecutas dos veces con **semillas distintas**?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Los mapas serán **distintos**: las islas pueden aparecer giradas, reflejadas o en otras posiciones. Lo que **sí** debería mantenerse es **qué puntos están juntos** (las islas en sí). Por eso, con t-SNE, fija siempre `random_state` y no le des importancia a "arriba/abajo" o "izquierda/derecha". Compruébalo:
</details>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, semilla in zip(axes, [0, 1, 2]):          # ⏳ ~20 s
    X_s = TSNE(perplexity=30, init="random", random_state=semilla).fit_transform(X_d[:500])
    mapa(ax, X_s, y_d[:500], f"random_state={semilla}")
plt.show()

> 📝 Usamos `init="random"` para que la diferencia se note. Con `init="pca"` (el valor por defecto) los mapas son más parecidos entre ejecuciones, porque todos parten de la misma "foto" de PCA.

---
## 4. 📖 Hiperparámetros: la perplejidad

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `perplexity` | ≈ cuántos vecinos considera cada punto | Visión más **global**: islas grandes, más estructura general | Visión muy **local**: muchas islitas, aparecen grupos falsos | 30 (prueba entre 5 y 50; debe ser menor que el número de datos) |
| `max_iter` | Cuántos pasos de optimización | Más tiempo, mapa más "asentado" | Mapa a medio hacer | 1000 (subir si el mapa se ve "apelotonado") |
| `learning_rate` | Tamaño de cada paso | — | — | `"auto"` |
| `init` | Posición inicial | — | — | `"pca"` (más estable) |
| `random_state` | Semilla | — | — | Fíjala siempre |
| `metric` | Cómo se mide la distancia original | — | — | `"euclidean"`; `"cosine"` para texto o embeddings |

### 🔮 Ejercicio 4
Probaremos `perplexity` = **2**, **30** y **100** con 500 dígitos. ¿Qué gráfico tendrá **más** islas pequeñas? ¿Cuál se parecerá más a una "nube" con grupos pegados?

✍️ *Mi apuesta:*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, perp in zip(axes, [2, 30, 100]):          # ✏️ prueba 5 o 50   ⏳ ~30 s
    X_p = TSNE(perplexity=perp, random_state=0).fit_transform(X_d[:500])
    mapa(ax, X_p, y_d[:500], f"perplexity={perp}")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **`perplexity=2`**: cada punto solo se preocupa de 1–2 vecinos. El mapa se rompe en **muchas islitas** y hilos; un mismo dígito queda partido en varios pedazos. Muy "local".
- **`perplexity=30`**: el punto justo: una isla limpia por dígito.
- **`perplexity=100`**: visión más **global** (¡cada punto mira a 100 de 500 vecinos!): las islas siguen ahí, pero más estiradas y difusas, y algunas se deforman o se parten.

Moraleja: **nunca saques conclusiones de un solo t-SNE**. Prueba varias perplejidades; lo que aparece en todas es real.
</details>

---
## 5. 📖 ¿Qué pasa si… lees el mapa al pie de la letra? Las tres trampas

### 🔮 Ejercicio 5 — Trampa 1: el tamaño de las islas
Generamos **dos** grupos de 300 puntos en 10 dimensiones: uno **muy compacto** (dispersión 1) y otro **muy disperso** (dispersión 10, ¡diez veces más extendido!). En el mapa de t-SNE, ¿cuál se verá más grande?

✍️ *Mi apuesta:*

In [ ]:
X_tam, y_tam = make_blobs(n_samples=[300, 300], n_features=10, centers=[[0] * 10, [60] * 10],
                          cluster_std=[1, 10], random_state=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
mapa(axes[0], PCA(n_components=2).fit_transform(X_tam), y_tam, "PCA (respeta tamaños)", cmap="coolwarm")
mapa(axes[1], TSNE(perplexity=30, random_state=0).fit_transform(X_tam), y_tam, "t-SNE", cmap="coolwarm")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

En **PCA** el grupo disperso (rojo) se ve mucho más grande, como en la realidad. En **t-SNE** los dos grupos se ven de **tamaño parecido**. t-SNE "ajusta la lupa" a la densidad de cada zona: los grupos dispersos se encogen y los compactos se expanden.

👉 **Trampa 1: en t-SNE, el tamaño de una isla no dice qué tan disperso es el grupo.** Como en la boda: todas las mesas son del mismo tamaño.
</details>

### 🔮 Ejercicio 6 — Trampa 2: grupos que no existen
Ahora datos **sin ninguna estructura**: 500 puntos al azar en 10 dimensiones, una sola "nube" uniforme. ¿Qué mostrará t-SNE con una perplejidad muy baja (2)? ¿Y con 30?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(0)
X_azar = rng.normal(size=(500, 10))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, perp in zip(axes, [2, 30]):
    mapa(ax, TSNE(perplexity=perp, random_state=0).fit_transform(X_azar), "steelblue", f"Ruido puro · perplexity={perp}")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Con **`perplexity=2`** aparecen grumos, hilos y "grupitos" que parecen estructura… ¡pero son ruido puro! Con **30** se ve una nube redonda sin grupos, que es la verdad.

👉 **Trampa 2: con perplejidad baja, t-SNE puede inventar grupos.** Igual que K-means (Lección 8.1), t-SNE no te avisa cuando no hay nada. Si sospechas de un grupo, compruébalo en los datos originales.
</details>

### 📖 Trampa 3: las distancias entre islas
Lo vimos con la boda: en t-SNE, **dos islas cercanas no son necesariamente más parecidas** que dos islas lejanas. Las posiciones relativas de las islas dependen de la perplejidad, de la semilla y de la inicialización. A veces coinciden con la realidad (con `init="pca"` suele conservarse algo del orden global), pero **no está garantizado**.

### ✋ Ejercicio 7 — ¿Qué puedes concluir?
Un analista te muestra un t-SNE de clientes y dice tres cosas. ¿Cuáles son conclusiones válidas?
1. "Hay 4 islas bien separadas: probablemente hay 4 tipos de clientes."
2. "La isla de los clientes jóvenes es enorme: son el grupo más variado."
3. "La isla VIP está lejísimos de la isla de ofertas: son los grupos más distintos entre sí."

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Razonable como hipótesis**, sobre todo si las 4 islas aparecen con varias perplejidades y semillas. Habría que confirmarlo agrupando los datos originales (Lecciones 8.2–8.3).
2. **No válida:** trampa 1. El tamaño de las islas no refleja la variedad del grupo.
3. **No válida:** trampa 3. Para comparar grupos, compara sus **perfiles** (promedios de cada columna) en los datos originales.
</details>

### 📖 Otras dos limitaciones prácticas
- **No tiene `transform`:** si llega un cliente nuevo, no puedes "ubicarlo" en el mapa existente; hay que recalcular todo.
- **Es lento** con muchos datos o muchas columnas. Truco estándar: primero **PCA a ~50 componentes** (Lección 8.4) y luego t-SNE sobre esos 50.

---
## 6. 📖 UMAP: el primo rápido de t-SNE

**UMAP** (*Uniform Manifold Approximation and Projection*, 2018) persigue la misma meta: un mapa 2D donde los vecinos queden juntos. Su intuición:

1. Construye una **red de amistades**: une cada punto con sus `n_neighbors` vecinos más cercanos.
2. Busca un dibujo en 2D que respete esa red lo mejor posible (los amigos cerca, sin enredar los hilos).

Comparado con t-SNE:

| | t-SNE | UMAP |
|---|---|---|
| Velocidad | Lento con > ~10,000 puntos | **Mucho más rápido**; aguanta millones |
| Estructura global | Poco confiable | **Algo mejor** conservada (aunque tampoco perfecta) |
| Datos nuevos | No tiene `transform` | **Sí** tiene `transform` |
| Perillas principales | `perplexity` | `n_neighbors`, `min_dist` |
| Disponible en | scikit-learn | Librería aparte: `umap-learn` |

| Hiperparámetro (UMAP) | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_neighbors` | Cuántos vecinos forman la red de cada punto (≈ la perplejidad de t-SNE) | Visión más global | Visión más local; puede romper grupos | 15 (prueba 5–50) |
| `min_dist` | Qué tan apretados pueden quedar los puntos en el mapa | Islas más esponjosas y repartidas | Islas muy compactas (útil si luego quieres agruparlas) | 0.1 |
| `metric` | Distancia original | — | — | `"euclidean"`; `"cosine"` para texto |

> ⚠️ UMAP **no está instalado** en el entorno de este curso. El código de abajo es **para Colab** (o para tu computadora después de `pip install umap-learn`). No lo ejecutes aquí.

```python
# 🧪 Para Colab: !pip install umap-learn
import umap
from sklearn.datasets import load_digits

X_d, y_d = load_digits(return_X_y=True)

reductor = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=0)
X_umap = reductor.fit_transform(X_d)              # mapa 2D, como t-SNE pero más rápido

import matplotlib.pyplot as plt
plt.scatter(*X_umap.T, c=y_d, cmap="tab10", s=6); plt.title("UMAP"); plt.show()

# A diferencia de t-SNE, UMAP puede ubicar datos nuevos en el mapa ya aprendido:
nuevos_2d = reductor.transform(X_d[:5])
```

### ✋ Ejercicio 8
Tu empresa tiene **2 millones** de clientes y quiere un mapa que se actualice cada semana **agregando** a los clientes nuevos sin redibujar todo. ¿t-SNE o UMAP? ¿Por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**UMAP**, por dos razones: (1) es mucho más rápido con millones de puntos, y (2) tiene `transform`, así que puede ubicar a los clientes nuevos en el mapa existente. t-SNE tendría que recalcular el mapa completo cada semana, y además el mapa nuevo podría salir girado o reorganizado.

(Aun así, con 2 millones de puntos, lo habitual es dibujar una **muestra** representativa: un gráfico con 2 millones de puntos es una mancha.)
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma / tentación | Por qué es un problema | Qué hacer |
|---|---|---|
| "Esta isla es enorme, ese grupo es muy variado" | t-SNE iguala los tamaños (trampa 1) | Medir la dispersión en los datos originales |
| "Estas dos islas están lejos, son muy distintas" | Las distancias entre islas no son fiables (trampa 3) | Comparar perfiles en los datos originales |
| Ver grupos con perplejidad 2–5 | Pueden ser inventados (trampa 2) | Probar varias perplejidades (5–50) y semillas |
| Usar las coordenadas de t-SNE como columnas para un modelo | No tiene `transform` y deforma distancias | Usar PCA (o UMAP con cuidado) para eso |
| Agrupar (K-means, DBSCAN) **sobre** el mapa de t-SNE y reportar esos grupos | Los grupos heredan las deformaciones del mapa | Agrupar en los datos originales (o en PCA) y usar t-SNE solo para **dibujarlos** |
| Tarda muchísimo | Muchos datos o columnas | PCA a 50 componentes antes; muestrear; usar UMAP |
| Mapa "apelotonado" en una bola | Pocas iteraciones | Subir `max_iter` |

### 🔮 Ejercicio 9 — Escala sin escalar
Si las columnas de tus datos tienen escalas muy distintas (pesos vs. años) y no escalas, ¿le afecta a t-SNE como a K-means y PCA? Piensa en el paso 1 de la sección 3.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

**Sí.** t-SNE empieza calculando **distancias** en el espacio original para decidir quién es vecino de quién (paso 1). Si una columna tiene números enormes, dominará esas distancias y los "vecinos" serán solo vecinos en esa columna. Regla de todo el módulo: **si hay distancias, escala primero** (salvo columnas en la misma unidad, como los píxeles).
</details>

---
## 8. 📋 Ficha del modelo

| | t-SNE | UMAP |
|---|---|---|
| **Tipo** | No supervisado: reducción de dimensionalidad **no lineal**, para visualizar | Igual |
| **Intuición** | Plano de mesas de una boda: los conocidos juntos | Red de amistades desenredada en un plano |
| **Cómo funciona** | Convierte distancias en "probabilidades de ser vecinos" y mueve los puntos en 2D hasta imitarlas | Construye un grafo de vecinos y busca un dibujo 2D que lo respete |
| **Hiperparámetros clave** | `perplexity`, `max_iter`, `random_state` | `n_neighbors`, `min_dist`, `metric` |
| **✅ Fortalezas** | Revela grupos que PCA no muestra · Mapas muy claros | Más rápido · Conserva algo más de estructura global · Tiene `transform` |
| **❌ Debilidades** | Lento · Sin `transform` · Tamaños y distancias entre grupos sin significado · Cambia con la semilla | Las mismas trampas de lectura (en menor grado) · Librería aparte |
| **Úsalo cuando…** | Quieres **ver** si hay grupos en datos de muchas columnas | Igual, sobre todo con muchos datos o si necesitas ubicar datos nuevos |
| **Evítalo cuando…** | Necesitas medir distancias, tamaños o usar el resultado como entrada de un modelo | Igual: para preprocesar, PCA suele ser más seguro |

## 9. 🧠 Resumen en 5 puntos
1. t-SNE y UMAP dibujan datos de muchas columnas en 2D **conservando los vecinos**; PCA conserva la **varianza global**.
2. Revelan grupos que PCA no puede mostrar, pero son **solo para visualizar**.
3. La **perplejidad** (t-SNE) y `n_neighbors` (UMAP) regulan cuántos vecinos mira cada punto: bajo = local (y grupos falsos), alto = global.
4. Las tres trampas: el **tamaño** de las islas, las **distancias** entre islas y los **grupos inventados** con perplejidad baja.
5. Fija la semilla, prueba varias perplejidades, escala los datos y **confirma** cualquier hallazgo en los datos originales.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** ¿Qué conserva mejor t-SNE?
- a) Las distancias entre grupos · b) Los vecinos de cada punto · c) El tamaño de los grupos · d) La varianza total

**2.** Quieres usar la reducción de dimensionalidad como **entrada** de un clasificador. ¿Qué eliges?
- a) t-SNE · b) PCA · c) Cualquiera de los dos da igual · d) Ninguno puede

**3.** Con `perplexity=3` ves 12 grupitos; con 30 y 50 ves 4 grupos. ¿Qué concluyes?
- a) Hay 12 grupos · b) Probablemente hay ~4 grupos; los 12 pueden ser artefactos · c) t-SNE está roto · d) Hay 50 grupos

**4.** Ejecutas t-SNE dos veces con semillas distintas y las islas aparecen en otras posiciones. ¿Es un problema?
- a) Sí, los datos están mal · b) No, es normal; importa qué puntos quedan juntos · c) Sí, hay que usar UMAP · d) Sí, hay que subir la perplejidad

**5.** ¿Qué ventaja tiene UMAP sobre t-SNE?
- a) Es determinista siempre · b) Es más rápido y tiene `transform` · c) Conserva perfectamente las distancias · d) No tiene hiperparámetros

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Los vecinos (estructura local). Las otras tres son justo lo que sacrifica.
2. **b)** PCA tiene `transform`, es determinista y no deforma. t-SNE es solo para dibujar.
3. **b)** Lo que es estable entre perplejidades razonables (5–50) es más creíble; con perplejidad muy baja aparecen grupos falsos.
4. **b)** Las posiciones, giros y reflejos no significan nada.
5. **b)** Más rápido y con `transform`. (Tampoco conserva perfectamente las distancias.)
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un investigador publica un t-SNE de células y escribe: "El grupo B está entre A y C, lo que sugiere que B es una etapa intermedia en el desarrollo de A a C." ¿Qué le dirías? ¿Cómo podría reforzar su conclusión?

**B. Caso de análisis 💼** Una plataforma de música tiene, para cada canción, un vector de 128 números que resume su sonido. El equipo de producto quiere "un mapa de toda la música para que los usuarios exploren géneros". ¿Qué técnica usarías, con qué precauciones, y qué **no** le prometerías al equipo?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Que la posición relativa de las islas en t-SNE no es fiable (trampa 3): "B en medio" podría ser un accidente del mapa. Para reforzarlo: repetir con varias perplejidades y semillas (¿B sigue en medio?), comparar con PCA o UMAP (que conservan algo más la estructura global), y sobre todo medir en los datos originales si los perfiles de B son de verdad intermedios entre A y C.

**B.** **UMAP** (o t-SNE sobre una muestra) tras escalar o normalizar los vectores, probablemente con `metric="cosine"`, que es la habitual para este tipo de vectores. Precauciones: probar varios `n_neighbors`, fijar la semilla y validar con personas (¿las islas corresponden a géneros reconocibles?). **No prometería** que la distancia en el mapa mida "qué tan distintos son dos géneros" ni que el tamaño de una isla indique la variedad del género. Ventaja de UMAP aquí: puede ubicar canciones nuevas en el mapa con `transform`.
</details>

### 10.3 🧩 Práctica integradora: "¿Cuántos tipos de usuarios tiene la app?"

Una app de ejercicio registra **20 métricas** por usuario (minutos por semana, tipo de entrenamiento, horario, ritmo cardiaco promedio…), ya escaladas. El equipo sospecha que hay "tipos" de usuarios, pero con 20 columnas nadie ve nada.

Lo harás en 5 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos.** (Son simulados: hay una respuesta "real" escondida en `tipo_real`, que solo usaremos al final para comprobar.)

In [ ]:
X_app, tipo_real = make_blobs(n_samples=[400, 300, 200, 150, 100, 50], n_features=20,
                              cluster_std=[1, 2.5, 1, 1, 0.5, 1], center_box=(-3, 3), random_state=5)
print("Usuarios × métricas:", X_app.shape)

**Paso 2 🔮 — Primero PCA.** ¿Crees que se verán claramente los grupos en 2D con PCA? ¿Cuántos cuentas?

✍️ *Mi apuesta:*

In [ ]:
pca_app = PCA(n_components=2).fit(X_app)
fig, ax = plt.subplots(figsize=(6, 5))
mapa(ax, pca_app.transform(X_app), "gray", f"PCA ({pca_app.explained_variance_ratio_.sum():.0%} de la varianza)")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Los dos primeros componentes solo capturan el **42 %** de la varianza. Se intuyen 4–5 zonas, y una de ellas es una mancha grande y difusa donde podría haber uno o dos grupos **superpuestos**. No queda claro cuántos tipos hay.
</details>

**Paso 3 ✏️ — Ahora t-SNE.** Ejecuta con la perplejidad por defecto y cuenta las islas. Luego cambia `PERPLEJIDAD` a 5 y a 50: ¿cambia el número de islas?

In [ ]:
PERPLEJIDAD = 30     # ✏️ prueba 5 y 50
X_app_tsne = TSNE(perplexity=PERPLEJIDAD, random_state=0).fit_transform(X_app)
fig, ax = plt.subplots(figsize=(6, 5))
mapa(ax, X_app_tsne, "gray", f"t-SNE · perplexity={PERPLEJIDAD}")
plt.show()

✍️ *¿Cuántas islas ves? ¿Se mantienen con otras perplejidades?*

<details><summary><b>👀 Ver respuesta</b></summary>

Se ven **6 islas** limpias, y siguen siendo 6 con perplejidad 5 y 50 (con 5, las islas se ven más "esponjosas"). Que sean estables entre perplejidades es buena señal: probablemente hay **6 tipos de usuarios**, ¡dos más de los que dejaba ver PCA!

¿Qué tan grande es cada grupo y qué tan variado? Por el tamaño de las islas no lo sabemos (trampa 1)… Paso 4.
</details>

**Paso 4 🔮 — Comprueba en los datos originales.** Agrupamos con K-means **en las 20 columnas originales** (no en el mapa), con k = 6, y miramos el tamaño de cada grupo. Luego coloreamos el mapa de t-SNE con esos grupos. ¿Coincidirán con las islas? ¿Cuál será el grupo más grande?

✍️ *Mi apuesta:*

In [ ]:
grupos_app = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(X_app)
print("Usuarios por grupo:", np.bincount(grupos_app))
print("Coincidencia con los tipos reales:")
print(pd.crosstab(tipo_real, grupos_app, rownames=["tipo real"], colnames=["grupo K-means"]))

fig, ax = plt.subplots(figsize=(6, 5))
mapa(ax, X_app_tsne, grupos_app, "t-SNE coloreado con los grupos de K-means (20 columnas)")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

Cada isla de t-SNE corresponde a un grupo de K-means, y casi coinciden con los tipos reales (solo unos 15 usuarios del tipo más disperso se asignan a otro grupo). Los tamaños son **muy distintos**: ≈**400, 290, 200, 150, 105 y 55** usuarios. Además, el grupo de ≈300 es en realidad **2.5 veces más disperso** que los demás, cosa que el mapa no deja ver: su isla se ve parecida a la del grupo de 400 (trampa 1).

Es el flujo de trabajo correcto: **t-SNE para descubrir y dibujar, el agrupamiento en los datos originales para medir.**
</details>

**Paso 5 ✋ — El informe.** Escribe dos frases para el equipo de producto: ¿cuántos tipos de usuarios hay y qué parte del gráfico **no** deben interpretar?

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"Los datos muestran 6 tipos de usuarios, de tamaños muy distintos: dos grandes (≈400 y ≈300 usuarios, este último muy variado por dentro), tres medianos (≈200, ≈150 y ≈100) y uno pequeño de ≈50 que vale la pena estudiar. En el mapa, el tamaño de cada isla y la distancia entre islas no significan nada: para comparar grupos usaremos sus métricas promedio."*

El siguiente paso natural sería **perfilar** cada grupo (promedio de cada métrica), como en la Lección 8.2.
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *How to Use t-SNE Effectively* (Distill.pub): juega con la perplejidad y descubre las trampas por ti mismo. Imprescindible.
- 🎮 **Interactivo:** *Understanding UMAP* (Google PAIR): comparación visual de UMAP y t-SNE.
- 📺 **Video:** StatQuest, "t-SNE, Clearly Explained" y "UMAP Dimension Reduction, Main Ideas" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Manifold learning*; documentación de `umap-learn`.

---
⬅️ **Anterior:** [8.4 Reducción de dimensionalidad: PCA, la mejor "foto" de los datos](8.4-pca.ipynb) · ➡️ **Siguiente:** [8.6 Detección de anomalías: Isolation Forest y compañía](8.6-deteccion-de-anomalias.ipynb)